In [1]:
!pip install rasterio geopandas ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 423.0 kB/s eta 0:00:00


In [15]:
import os, gc, cv2, time, json, torch, rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import psutil

from datetime import datetime
from collections import defaultdict
from tqdm import tqdm

from PIL import Image
from rasterio.windows import Window
from rasterio.transform import from_bounds
from shapely.geometry import Point, Polygon

from ultralytics import YOLO
from IPython.display import Image as IPyImage

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# ===========================================================================
# CONFIGURATION
# ===========================================================================
CONFIG = {
    # ── Tiling ──────────────────────────────────────────────────────────────
    'tile_size':      640,
    'overlap':        128,
    'resolution':     1.0,   # 1.0 = native resolution; 0.5 = half-scale
    'batch_size':     16,

    # ── YOLO ────────────────────────────────────────────────────────────────
    'conf_threshold': 0.10,
    'iou_threshold':  0.10,
    'max_det':        5000,

    # ── NMS (post-YOLO deduplication) ───────────────────────────────────────
    'apply_nms':      True,
    'nms_iou':        0.30,

    # ── Paths ────────────────────────────────────────────────────────────────
    'model_path':  '/content/drive/MyDrive/AGRI/TreeCrown_Segmentation/models/detection/yolo11n-100epoch-v4.pt',
    'image_path':  '/content/drive/MyDrive/AGRI/TreeCrown_Segmentation/Screenshot 2026-03-31 141747.png',
    'output_dir':  '/content/drive/MyDrive/AGRI/Detection',
    'image_name':  'tree_detection',

    # ── Visualization ────────────────────────────────────────────────────────
    'max_viz_dimension': 4000,
}

In [20]:
MEM_LIMIT_GB = 8.0   # full-image RAM cache threshold


# ===========================================================================
# HELPERS
# ===========================================================================

def _gpu_available():
    return torch.cuda.is_available()


def _setup_gpu_memory():
    if not _gpu_available():
        print("  No GPU detected — running on CPU")
        return
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {total_gb:.1f} GB total")


# ===========================================================================
# STATE
# ===========================================================================

class DetectionState:
    """Holds all data that flows through the detection pipeline."""

    def __init__(self, image_path: str, model, config: dict):
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")
        self.image_path = image_path
        self.model      = model
        self.config     = config

        self.original_dims = None   # (width, height)
        self.scaled_dims   = None   # after resolution scaling
        self.transform     = None
        self.crs           = None

        self.boxes_list    = []     # raw YOLO detections
        self.detection_info = []    # enriched per-detection dicts

        self._full_image_cache = None
        self._cache_attempted  = False

        self.start_time     = time.time()
        self.start_cpu_time = time.process_time()
        self.stage_times    = {}
        self.stage_memory   = {}
        self.process        = psutil.Process()

    # ── Image cache ─────────────────────────────────────────────────────────

    def _maybe_cache_full_image(self):
        if self._cache_attempted:
            return
        self._cache_attempted = True
        if self.original_dims is None:
            return
        orig_w, orig_h = self.original_dims
        image_gb = orig_w * orig_h * 3 / 1024**3
        if image_gb > MEM_LIMIT_GB:
            print(f"  Image {image_gb:.1f} GB > limit {MEM_LIMIT_GB} GB — windowed reads only")
            return
        print(f"  Caching full image ({image_gb:.2f} GB) into RAM…")
        try:
            with rasterio.open(self.image_path) as src:
                data = src.read([1, 2, 3]) if src.count >= 3 else cv2.cvtColor(src.read(1), cv2.COLOR_GRAY2RGB)
                if data.ndim == 3 and data.shape[0] == 3:
                    data = np.transpose(data, (1, 2, 0))
            self._full_image_cache = np.clip(data, 0, 255).astype(np.uint8)
            print(f"  ✓ Cached ({self._full_image_cache.nbytes / 1024**3:.2f} GB)")
        except MemoryError:
            print("  MemoryError — windowed reads only")
            self._full_image_cache = None

    def read_crop(self, x1: int, y1: int, x2: int, y2: int) -> np.ndarray:
        orig_w, orig_h = self.original_dims
        x1 = max(0, min(x1, orig_w));  y1 = max(0, min(y1, orig_h))
        x2 = max(x1, min(x2, orig_w)); y2 = max(y1, min(y2, orig_h))
        if x2 <= x1 or y2 <= y1:
            return np.zeros((0, 0, 3), dtype=np.uint8)
        if self._full_image_cache is not None:
            return self._full_image_cache[y1:y2, x1:x2].copy()
        window = Window(x1, y1, x2 - x1, y2 - y1)
        try:
            with rasterio.open(self.image_path) as src:
                if src.count >= 3:
                    data = src.read([1, 2, 3], window=window)
                    data = np.transpose(data, (1, 2, 0))
                else:
                    data = src.read(1, window=window)
                    data = cv2.cvtColor(data, cv2.COLOR_GRAY2RGB)
            return np.clip(data, 0, 255).astype(np.uint8)
        except Exception as e:
            print(f"  ⚠ read_crop failed ({e})")
            return np.zeros((y2 - y1, x2 - x1, 3), dtype=np.uint8)

    # ── Performance logging ──────────────────────────────────────────────────

    def log_performance(self, stage_name: str):
        mem     = self.process.memory_info()
        proc_mb = mem.rss / 1024**2
        sys_m   = psutil.virtual_memory()
        wall    = time.time()         - self.start_time
        cpu     = time.process_time() - self.start_cpu_time
        self.stage_memory[stage_name] = {
            'process_ram_mb':          proc_mb,
            'system_ram_gb':           sys_m.used      / 1024**3,
            'system_ram_percent':      sys_m.percent,
            'system_ram_available_gb': sys_m.available / 1024**3,
        }
        self.stage_times[stage_name] = {
            'wall_time': wall, 'cpu_time': cpu,
            'timestamp': datetime.now().strftime('%H:%M:%S'),
        }
        vram_str = ''
        if _gpu_available():
            used = torch.cuda.memory_allocated(0) / 1024**3
            res  = torch.cuda.memory_reserved(0)  / 1024**3
            vram_str = f"\n   VRAM        : {used:.2f} GB alloc / {res:.2f} GB reserved"
        print(f"\n📊 {stage_name}")
        print(f"   Process RAM : {proc_mb:.0f} MB ({proc_mb/1024:.2f} GB)")
        print(f"   System RAM  : {sys_m.used/1024**3:.2f} GB ({sys_m.percent:.1f}% used)"
              f" — {sys_m.available/1024**3:.2f} GB free")
        print(f"   Time        : {wall:.1f}s wall | {cpu:.1f}s CPU" + vram_str)

    def print_performance_summary(self):
        print("\n" + "="*80)
        print(" "*30 + "PERFORMANCE SUMMARY")
        print("="*80)
        print(f"\n⏱️  TIMING:")
        print(f"{'Stage':<42} {'Wall':>10}  {'CPU':>10}")
        print("-"*80)
        for s, t in self.stage_times.items():
            print(f"{s:<42} {t['wall_time']:>8.2f}s  {t['cpu_time']:>8.2f}s")
        tw = time.time()         - self.start_time
        tc = time.process_time() - self.start_cpu_time
        print("-"*80)
        print(f"{'TOTAL':<42} {tw:>8.2f}s  {tc:>8.2f}s")
        print(f"\n💾 MEMORY:")
        print(f"{'Stage':<42} {'Proc RAM':>10}  {'Sys RAM':>10}")
        print("-"*80)
        for s, m in self.stage_memory.items():
            print(f"{s:<42} {m['process_ram_mb']/1024:>8.2f} GB"
                  f"  {m['system_ram_gb']:>6.2f} GB ({m['system_ram_percent']:.0f}%)")
        if self.stage_memory:
            pp = max(m['process_ram_mb'] for m in self.stage_memory.values()) / 1024
            ps = max(m['system_ram_gb']  for m in self.stage_memory.values())
            print("-"*80)
            print(f"{'PEAK':<42} {pp:>8.2f} GB  {ps:>6.2f} GB")
            print(f"\n📈 SUMMARY:")
            print(f"   Total time       : {tw/60:.1f} min")
            print(f"   Peak process RAM : {pp:.2f} GB")
            print(f"   YOLO detections  : {len(self.boxes_list)}")
        print("="*80)

    def __repr__(self):
        return f"DetectionState(boxes={len(self.boxes_list)})"


# ===========================================================================
# TRANSITIONS
# ===========================================================================

class StateTransition:
    def __call__(self, state: DetectionState) -> DetectionState:
        raise NotImplementedError


class LoadMetadataTransition(StateTransition):
    """Open the raster and record its dimensions, transform and CRS."""

    def __call__(self, state):
        print("Loading image metadata…")
        state.log_performance("1. Metadata")
        try:
            with rasterio.open(state.image_path) as src:
                state.original_dims = (src.width, src.height)
                state.transform     = src.transform
                state.crs           = src.crs
                nb                  = src.count
            print(f"  ✓ {state.original_dims[0]}×{state.original_dims[1]} px  "
                  f"bands={nb}  "
                  f"({state.original_dims[0]*state.original_dims[1]*3/1024**3:.2f} GB raw RGB)")
        except Exception:
            from PIL import Image as PILImage
            with PILImage.open(state.image_path) as img:
                w, h = img.size
            state.original_dims = (w, h)
            state.transform     = from_bounds(0, 0, w, h, w, h)
            state.crs           = None
        state._maybe_cache_full_image()
        return state


class CalculateScaledDimsTransition(StateTransition):
    """Compute the working (scaled) dimensions based on resolution factor."""

    def __call__(self, state):
        r    = state.config['resolution']
        w, h = state.original_dims
        state.scaled_dims = (int(w * r), int(h * r))
        print(f"  ✓ Scaled dims: {state.scaled_dims[0]}×{state.scaled_dims[1]}  "
              f"(resolution={r})")
        return state


class YOLOStreamingTransition(StateTransition):
    """Tile the image and run YOLO batches, storing raw bounding boxes."""

    def __call__(self, state):
        state.log_performance("2. YOLO Detection")
        cfg        = state.config
        tile_size  = cfg['tile_size']
        overlap    = cfg['overlap']
        conf       = cfg['conf_threshold']
        iou        = cfg['iou_threshold']
        max_det    = cfg['max_det']
        resolution = cfg['resolution']
        batch_size = cfg.get('batch_size', 16)

        new_w, new_h = state.scaled_dims
        step         = tile_size - overlap

        tile_positions = [
            (x, y, min(x + tile_size, new_w), min(y + tile_size, new_h))
            for y in range(0, new_h, step)
            for x in range(0, new_w, step)
        ]
        print(f"  {len(tile_positions)} tiles  batch={batch_size}")
        device = 'cuda' if _gpu_available() else 'cpu'

        state.boxes_list = self._run_batches(
            state, tile_positions, tile_size,
            conf, iou, max_det, resolution, batch_size, device
        )
        print(f"  ✓ {len(state.boxes_list)} raw boxes detected")
        return state

    def _run_batches(self, state, tile_positions, tile_size,
                     conf, iou, max_det, resolution, batch_size, device):
        all_boxes = []
        orig_w, orig_h = state.original_dims

        for start in tqdm(range(0, len(tile_positions), batch_size), desc="YOLO"):
            batch  = tile_positions[start : start + batch_size]
            images, meta = [], []

            for x, y, x_end, y_end in batch:
                # Map scaled-space tile coords back to original pixel coords
                ox  = int(np.clip(x     / resolution, 0, orig_w - 1))
                oy  = int(np.clip(y     / resolution, 0, orig_h - 1))
                ox2 = int(np.clip(x_end / resolution, 0, orig_w))
                oy2 = int(np.clip(y_end / resolution, 0, orig_h))

                tile     = state.read_crop(ox, oy, ox2, oy2)
                win_w, win_h = x_end - x, y_end - y

                # Ensure the tile matches the expected window size
                if tile.shape[:2] != (win_h, win_w):
                    tile = cv2.resize(tile, (win_w, win_h))

                # Scale to model input size and record the scale factors
                if tile.shape[:2] != (tile_size, tile_size):
                    sx = win_w / tile_size
                    sy = win_h / tile_size
                    tile = cv2.resize(tile, (tile_size, tile_size))
                else:
                    sx = sy = 1.0

                images.append(np.ascontiguousarray(tile))
                meta.append((x, y, sx, sy))

            results = state.model.predict(
                source=images, conf=conf, iou=iou,
                max_det=max_det, verbose=False, device=device
            )

            for result, (tx, ty, sx, sy) in zip(results, meta):
                if result.boxes is not None and len(result.boxes):
                    boxes = result.boxes.xyxy.cpu().numpy()
                    confs = result.boxes.conf.cpu().numpy()
                    for i in range(len(boxes)):
                        x1, y1_b, x2, y2_b = boxes[i]
                        # Convert tile-local coords → original image pixel coords
                        gx1 = int((x1 * sx + tx) / resolution)
                        gy1 = int((y1_b * sy + ty) / resolution)
                        gx2 = int((x2 * sx + tx) / resolution)
                        gy2 = int((y2_b * sy + ty) / resolution)
                        cx  = (gx1 + gx2) / 2
                        cy  = (gy1 + gy2) / 2
                        all_boxes.append({
                            'bbox':       [gx1, gy1, gx2, gy2],
                            'center_x':   cx,
                            'center_y':   cy,
                            'confidence': float(confs[i]),
                        })

            del images, results
            gc.collect()
            if _gpu_available():
                torch.cuda.empty_cache()

        return all_boxes


class NMSTransition(StateTransition):
    """Non-maximum suppression to remove duplicate detections across tiles."""

    def __call__(self, state):
        state.log_performance("3. NMS")
        if not state.config.get('apply_nms', True) or not state.boxes_list:
            return state
        iou_thr = state.config.get('nms_iou', 0.30)
        before  = len(state.boxes_list)
        state.boxes_list = self._nms(state.boxes_list, iou_thr)
        print(f"  NMS: {before} → {len(state.boxes_list)} boxes  (IoU threshold={iou_thr})")
        return state

    @staticmethod
    def _nms(boxes, iou_thr):
        boxes   = sorted(boxes, key=lambda b: b['confidence'], reverse=True)
        keep    = []
        removed = set()
        for i, bi in enumerate(boxes):
            if i in removed:
                continue
            keep.append(bi)
            ai = bi['bbox']
            for j in range(i + 1, len(boxes)):
                if j in removed:
                    continue
                aj = boxes[j]['bbox']
                ix1 = max(ai[0], aj[0]); iy1 = max(ai[1], aj[1])
                ix2 = min(ai[2], aj[2]); iy2 = min(ai[3], aj[3])
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2 - ix1) * (iy2 - iy1)
                    union = ((ai[2]-ai[0])*(ai[3]-ai[1]) +
                             (aj[2]-aj[0])*(aj[3]-aj[1]) - inter)
                    if union > 0 and inter / union > iou_thr:
                        removed.add(j)
        return keep


class BuildGeoDataFrameTransition(StateTransition):
    """Convert raw box list to a GeoDataFrame with geo-referenced points."""

    def __call__(self, state):
        state.log_performance("4. GeoDataFrame")
        print(f"  Building GeoDataFrame from {len(state.boxes_list)} boxes…")
        if not state.boxes_list:
            state.gdf = gpd.GeoDataFrame()
            return state

        points, info = [], []
        for b in tqdm(state.boxes_list, desc="Geo-points"):
            px, py = rasterio.transform.xy(
                state.transform, b['center_y'], b['center_x']
            )
            points.append(Point(float(px), float(py)))
            info.append({
                'center_x':   b['center_x'],
                'center_y':   b['center_y'],
                'confidence': b['confidence'],
                'bbox_x1':    b['bbox'][0],
                'bbox_y1':    b['bbox'][1],
                'bbox_x2':    b['bbox'][2],
                'bbox_y2':    b['bbox'][3],
            })

        state.gdf = gpd.GeoDataFrame(
            info, geometry=points,
            crs=state.crs if state.crs else 'EPSG:4326'
        )
        state.detection_info = info
        print(f"  ✓ {len(state.gdf)} geo-referenced tree detections")
        return state


class VisualizationTransition(StateTransition):
    """Render detections onto a downsampled background and save PNG."""

    def __call__(self, state):
        state.log_performance("5. Visualization")
        if not state.boxes_list:
            print("  No detections to visualize")
            return state

        cfg        = state.config
        output_dir = cfg['output_dir']
        max_dim    = cfg.get('max_viz_dimension', 4000)
        orig_w, orig_h = state.original_dims
        scale  = min(max_dim / orig_w, max_dim / orig_h, 1.0)
        out_w  = int(orig_w * scale)
        out_h  = int(orig_h * scale)

        os.makedirs(output_dir, exist_ok=True)

        # Read background at display resolution
        bg_tile = state.read_crop(0, 0, orig_w, orig_h)
        if bg_tile.size > 0:
            bg = cv2.resize(bg_tile, (out_w, out_h), interpolation=cv2.INTER_AREA)
        else:
            bg = np.zeros((out_h, out_w, 3), dtype=np.uint8)
        del bg_tile

        canvas = bg.copy()

        # Draw a dot at each tree center
        for b in tqdm(state.boxes_list, desc="Drawing points"):
            cx = int(b['center_x'] * scale); cy = int(b['center_y'] * scale)
            cv2.circle(canvas, (cx, cy), 4, (255, 80, 20), -1)

        # Overlay stats — white box legend (like SAM-style output)
        n     = len(state.boxes_list)
        avg_c = np.mean([b['confidence'] for b in state.boxes_list]) if n else 0

        title    = "TREE CROWN SEGMENTATION"
        line1    = f"Trees Detected:  {n:,}"
        line2    = f"Avg Confidence: {avg_c:.3f}"
        lines    = [line1, line2]

        font       = cv2.FONT_HERSHEY_SIMPLEX
        pad        = 12
        title_scale, title_thick = 0.75, 2
        text_scale,  text_thick  = 0.50, 1

        (tw, th), _ = cv2.getTextSize(title, font, title_scale, title_thick)
        line_sizes  = [cv2.getTextSize(l, font, text_scale, text_thick) for l in lines]
        box_w = max(tw, max(s[0][0] for s in line_sizes)) + pad * 2
        line_h = max(s[0][1] for s in line_sizes) + 6
        box_h = th + pad * 2 + len(lines) * line_h + pad

        # White filled rectangle
        cv2.rectangle(canvas, (10, 10), (10 + box_w, 10 + box_h), (255, 255, 255), -1)

        # Title
        cv2.putText(canvas, title, (10 + pad, 10 + pad + th),
                    font, title_scale, (0, 0, 0), title_thick, cv2.LINE_AA)

        # Stat lines
        y = 10 + pad + th + pad
        for line in lines:
            y += line_h
            cv2.putText(canvas, line, (10 + pad, y),
                        font, text_scale, (0, 0, 0), text_thick, cv2.LINE_AA)

        out_path = os.path.join(output_dir, f"{cfg.get('image_name','detection')}_detections.png")
        cv2.imwrite(out_path,
                    cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_PNG_COMPRESSION, 6])
        print(f"  ✓ Saved: {out_path}  ({os.path.getsize(out_path)/1024**2:.1f} MB)")
        del canvas, bg
        gc.collect()
        return state


class SaveResultsTransition(StateTransition):
    """Write CSV, GeoJSON and a summary JSON to the output directory."""

    def __call__(self, state):
        state.log_performance("6. Save Results")
        if not state.boxes_list:
            print("  No detections to save")
            return state

        output_dir = state.config['output_dir']
        name       = state.config.get('image_name', 'detection')
        os.makedirs(output_dir, exist_ok=True)

        # CSV of all detections
        csv_path = os.path.join(output_dir, f"{name}_detections.csv")
        gpd.GeoDataFrame(state.detection_info).to_csv(csv_path, index=False)
        print(f"  ✓ CSV: {csv_path}")

        # GeoJSON
        if state.gdf is not None and len(state.gdf) > 0 and state.crs is not None:
            geo_path = os.path.join(output_dir, f"{name}_detections.geojson")
            try:
                state.gdf.to_file(geo_path, driver='GeoJSON')
                print(f"  ✓ GeoJSON: {geo_path}")
            except Exception as e:
                print(f"  ⚠ GeoJSON failed: {e}")

        # Summary JSON
        confs = [b['confidence'] for b in state.boxes_list]
        stats = {
            'total_detections':    len(state.boxes_list),
            'average_confidence':  float(np.mean(confs)),
            'min_confidence':      float(np.min(confs)),
            'max_confidence':      float(np.max(confs)),
        }
        stats_path = os.path.join(output_dir, f"{name}_statistics.json")
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f"  ✓ Statistics: {stats_path}")

        total_time = time.time() - state.start_time
        print(f"  ✓ Done in {total_time/60:.1f} min")
        return state


# ===========================================================================
# PIPELINE
# ===========================================================================

class DetectionPipeline:
    def __init__(self, transitions):
        self.transitions = transitions

    def run(self, state: DetectionState) -> DetectionState:
        for i, t in enumerate(self.transitions):
            print(f"\n{'='*60}")
            print(f"[{i+1}/{len(self.transitions)}] {t.__class__.__name__}")
            print("="*60)
            state = t(state)
        return state


def create_detection_pipeline():
    return DetectionPipeline([
        LoadMetadataTransition(),
        CalculateScaledDimsTransition(),
        YOLOStreamingTransition(),
        NMSTransition(),
        BuildGeoDataFrameTransition(),
        VisualizationTransition(),
        SaveResultsTransition(),
    ])


# ===========================================================================
# ENTRY POINT
# ===========================================================================

_setup_gpu_memory()

# Enable performance optimisations
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

# Load YOLO model
yolo_model = YOLO(CONFIG['model_path'])
yolo_model.fuse()
if _gpu_available():
    yolo_model.half()

# Build state and run pipeline
state    = DetectionState(
    image_path=CONFIG['image_path'],
    model=yolo_model,
    config=CONFIG,
)
pipeline = create_detection_pipeline()
final    = pipeline.run(state)
final.print_performance_summary()

  No GPU detected — running on CPU
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

[1/7] LoadMetadataTransition
Loading image metadata…

📊 1. Metadata
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.46 GB (14.0% used) — 10.90 GB free
   Time        : 0.0s wall | 0.0s CPU
  ✓ 966×773 px  bands=4  (0.00 GB raw RGB)
  Caching full image (0.00 GB) into RAM…


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  ✓ Cached (0.00 GB)

[2/7] CalculateScaledDimsTransition
  ✓ Scaled dims: 966×773  (resolution=1.0)

[3/7] YOLOStreamingTransition

📊 2. YOLO Detection
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.46 GB (14.0% used) — 10.90 GB free
   Time        : 0.0s wall | 0.0s CPU
  4 tiles  batch=16


YOLO: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]


  ✓ 215 raw boxes detected

[4/7] NMSTransition

📊 3. NMS
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.47 GB (14.1% used) — 10.88 GB free
   Time        : 1.0s wall | 0.9s CPU
  NMS: 215 → 175 boxes  (IoU threshold=0.3)

[5/7] BuildGeoDataFrameTransition

📊 4. GeoDataFrame
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.47 GB (14.1% used) — 10.88 GB free
   Time        : 1.0s wall | 1.0s CPU
  Building GeoDataFrame from 175 boxes…


Geo-points: 100%|██████████| 175/175 [00:00<00:00, 15791.81it/s]


  ✓ 175 geo-referenced tree detections

[6/7] VisualizationTransition

📊 5. Visualization
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.47 GB (14.1% used) — 10.88 GB free
   Time        : 1.0s wall | 1.0s CPU


Drawing points: 100%|██████████| 175/175 [00:00<00:00, 189029.93it/s]


  ✓ Saved: /content/drive/MyDrive/AGRI/Detection/tree_detection_detections.png  (1.2 MB)

[7/7] SaveResultsTransition

📊 6. Save Results
   Process RAM : 1092 MB (1.07 GB)
   System RAM  : 1.47 GB (14.1% used) — 10.89 GB free
   Time        : 1.4s wall | 1.3s CPU
  ✓ CSV: /content/drive/MyDrive/AGRI/Detection/tree_detection_detections.csv
  ✓ Statistics: /content/drive/MyDrive/AGRI/Detection/tree_detection_statistics.json
  ✓ Done in 0.0 min

                              PERFORMANCE SUMMARY

⏱️  TIMING:
Stage                                            Wall         CPU
--------------------------------------------------------------------------------
1. Metadata                                    0.00s      0.00s
2. YOLO Detection                              0.05s      0.04s
3. NMS                                         0.96s      0.94s
4. GeoDataFrame                                0.97s      0.95s
5. Visualization                               0.99s      0.97s
6. Save Results        